# Final model · Stage 4 — scoring package

**Goal:** one engine that takes a *raw* loan application and returns the final decision JSON:

```
raw application -> preprocessor_v2.joblib -> final_model.txt -> calibrator.json -> decision_policy.json -> reasons -> JSON
                   (Preprocessing V2)        (Stage 2 model)    (Stage 2)          (Stage 3)               (Stage 2)
```

Runs in **one environment**: Python 3.14 with scikit-learn 1.9.0, the version that fitted the preprocessor.
Validation data only; **test sealed**. Plan: `IMPLEMENTATION_PLAN.md`.

## 1. Setup

In [1]:
import json, subprocess, sys, time, platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn, lightgbm as lgb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "engine").is_dir())
sys.path.insert(0, str(ROOT))
STAGE = ROOT / "models" / "final" / "04_scoring_package"
RESULTS = STAGE / "results"
RESULTS.mkdir(exist_ok=True)
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| lightgbm", lgb.__version__, "| pandas", pd.__version__)

Python 3.14.6 | scikit-learn 1.9.0 | lightgbm 4.7.0 | pandas 2.3.3


## 2. Build the bundle
`build_bundle.py` copies the four files into `bundle/` and writes `manifest.json` (checksums, versions, features, input fields).

In [2]:
run = subprocess.run([sys.executable, str(STAGE / "build_bundle.py")], capture_output=True, text=True, cwd=ROOT)
print(run.stdout, run.stderr)
assert run.returncode == 0
manifest = json.loads((STAGE / "bundle" / "manifest.json").read_text(encoding="utf-8"))
pd.DataFrame({"file": list(manifest["sha256"]), "sha256 (first 16)": [h[:16] for h in manifest["sha256"].values()],
              "MB": [round((STAGE / "bundle" / f).stat().st_size / 1e6, 2) for f in manifest["sha256"]]})

bundle: C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\models\final\04_scoring_package\bundle
  146 features from 65 raw input fields; 37 raw fields ignored
  preprocessor_v2.joblib         5.54 MB
  final_model.txt                4.81 MB
  calibrator.json                0.00 MB
  decision_policy.json           0.00 MB
 


,file,sha256 (first 16),MB
0,preprocessor_v2.joblib,58b16ab7b8b2d7d6,5.54
1,final_model.txt,2ce268265c9ca8b1,4.81
2,calibrator.json,3393d1aaa8a49cff,0.00
3,decision_policy.json,ac64b0af6d1ea72b,0.00


In [3]:
print("Environment recorded in the manifest:", manifest["environment"])
assert sklearn.__version__ == manifest["environment"]["scikit_learn"]

Environment recorded in the manifest: {'scikit_learn': '1.9.0', 'python_fitted': '3.14.6', 'lightgbm_trained': '4.6.0', 'lightgbm_verified': '4.7.0'}


## 3. The input contract
The preprocessor accepts 102 raw LendingClub columns, but the final model uses only the features built from **65 of them**. The other 37 are ignored. They are LendingClub's own risk outputs (grade, sub-grade, interest rate, instalment, excluded in version B), `zip_code` (dropped in Stage 2), joint-application and secondary-applicant fields, and bureau fields that were missing for most training loans.

In [4]:
groups = {"loan": ["loan_amnt", "term", "purpose", "title"],
          "applicant": ["annual_inc", "dti", "emp_title", "emp_length", "home_ownership", "verification_status", "addr_state"],
          "credit score / history": ["fico_range_low", "earliest_cr_line", "inq_last_6mths", "mths_since_recent_inq", "delinq_2yrs", "pub_rec",
                                     "pub_rec_bankruptcies", "mths_since_last_delinq"]}
fields = manifest["model_input_fields"]
groups["other credit-bureau fields"] = [f for f in fields if not any(f in g for g in groups.values())]
contract = pd.DataFrame([{"group": g, "fields": len(v), "examples": ", ".join(v[:8]) + (" ..." if len(v) > 8 else "")} for g, v in groups.items()])
print(f"{len(fields)} model input fields; ignored: {', '.join(manifest['ignored_raw_fields'][:8])} ... ({len(manifest['ignored_raw_fields'])} in total)")
contract

65 model input fields; ignored: int_rate, installment, grade, sub_grade, zip_code, fico_range_high, application_type, annual_inc_joint ... (37 in total)


,group,fields,examples
0,loan,4,"loan_amnt, term, purpose, title"
1,applicant,7,"annual_inc, dti, emp_title, emp_length, home_o..."
2,credit score / history,8,"fico_range_low, earliest_cr_line, inq_last_6mt..."
3,other credit-bureau fields,46,"mths_since_last_record, open_acc, revol_bal, r..."


## 4. Fresh-session test
Run as a **separate Python process**, so nothing from this notebook leaks into it. Among other checks, it reads the *original raw CSV rows* of 2,000 validation loans and requires the engine to reproduce Stage 3's PD, band, decision and expected loss exactly.

In [5]:
start = time.time()
run = subprocess.run([sys.executable, str(ROOT / "tests" / "engine_fresh_session_test.py")], capture_output=True, text=True, cwd=ROOT)
print(run.stdout[-6000:], run.stderr[-2000:])
print(f"({time.time() - start:.0f} s)")
report = json.loads((RESULTS / "f4_engine_test.json").read_text(encoding="utf-8"))
assert run.returncode == 0 and report["passed"]
pd.DataFrame(report["checks"])[["check", "required", "status"]]

PASS    engine loads in a fresh process; all bundle checksums match | 4 files, 0.2 s, scikit-learn 1.9.0
PASS    a tampered bundle file is refused | bundle file calibrator.json does not match its checksum in manifest.json
PASS    raw CSV rows of 2,000 validation loans reproduce Stage 3 exactly | max |PD diff| 0.0e+00, max |EL diff| $0.0e+00, bands equal True, decisions equal True; decisions: {'APPROVE': 1262, 'DECLINE': 538, 'REVIEW': 200}
PASS    single-application scoring agrees with batch scoring (20 loans) | max |diff| 5.0e-05 (PD is rounded to 4 decimals in the JSON); decisions equal True
PASS    sparse hand-written application -> complete JSON, no NaN | PD 0.1008, R3, APPROVE, EL $614.4, 47 missing fields listed
PASS    impossible inputs are rejected with a message | loan amount missing: loan_amnt (loan amount in USD) is required and must be a number; term of 48 months: term is required and must be 36 or 60 (months); FICO of 900: fico_range_low must be between 300 and 850; negati

,check,required,status
0,engine loads in a fresh process; all bundle ch...,True,PASS
1,a tampered bundle file is refused,True,PASS
2,"raw CSV rows of 2,000 validation loans reprodu...",True,PASS
3,single-application scoring agrees with batch s...,True,PASS
4,sparse hand-written application -> complete JS...,True,PASS
5,impossible inputs are rejected with a message,True,PASS
6,same input -> same output,True,PASS
7,higher FICO never raises PD; higher DTI never ...,True,PASS
8,one application scored in under 200 ms (median...,False,WARNING


## 5. The engine in use
The same `CreditRiskEngine` class the API will import.

In [6]:
from src.engine import ApplicationError, CreditRiskEngine
engine = CreditRiskEngine()
application = {
    "loan_amnt": 15000, "term": "60 months", "purpose": "debt_consolidation", "title": "Debt consolidation",
    "annual_inc": 48000, "dti": 27.5, "emp_title": "Sales associate", "emp_length": "2 years", "home_ownership": "RENT",
    "verification_status": "Source Verified", "addr_state": "FL",
    "fico_range_low": 665, "earliest_cr_line": "Jun-2011", "inq_last_6mths": 3, "revol_util": 88.0, "revol_bal": 14200,
    "open_acc": 11, "total_acc": 17, "acc_open_past_24mths": 7, "bc_open_to_buy": 600, "mort_acc": 0,
}
print(json.dumps(engine.score(application), indent=2))

{
  "model": {
    "name": "M3-B-mono-nozip",
    "version": "1.0.0"
  },
  "application_month": "Oct-2026",
  "probability_of_default": 0.6642,
  "risk_band": "R7",
  "risk_band_label": "severe risk",
  "decision": "DECLINE",
  "decision_rule": "PD >= 27.5%",
  "loan_amount": 15000.0,
  "term_months": 60,
  "expected_loss_usd": 6243.03,
  "expected_loss_rate": 0.4162,
  "reasons": [
    {
      "reason": "Loan term: 60 months",
      "impact": 0.6336
    },
    {
      "reason": "Job title (historical default rate of similar titles): 24.5%",
      "impact": 0.2277
    },
    {
      "reason": "Debt-to-income ratio: 27.5%",
      "impact": 0.2227
    },
    {
      "reason": "Credit score (FICO): 665",
      "impact": 0.2159
    }
  ],
  "strengths": [
    {
      "reason": "Months since the most recent credit inquiry: not reported",
      "impact": -0.0895
    },
    {
      "reason": "Bankcard utilisation: not reported",
      "impact": -0.0343
    }
  ],
  "missing_fields": [
    "d

In [7]:
print("Bad input is rejected with a message, not a crash:")
for bad in ({**application, "term": 48}, {**application, "fico_range_low": 950}, {"term": 36}):
    try:
        engine.score(bad)
    except ApplicationError as err:
        print("  ApplicationError:", err)

Bad input is rejected with a message, not a crash:
  ApplicationError: term is required and must be 36 or 60 (months)
  ApplicationError: fico_range_low must be between 300 and 850
  ApplicationError: loan_amnt (loan amount in USD) is required and must be a number


### Three real validation applicants, from their raw CSV rows
One per decision (picked at random from Stage 3's results). The actual outcome was never shown to the engine.

In [8]:
stage3 = pd.read_parquet(ROOT / "models" / "final" / "03_decision_layer" / "results" / "f3_validation_decisions.parquet")
picks = pd.concat([stage3[stage3["decision"] == d].sample(1, random_state=7) for d in ("APPROVE", "REVIEW", "DECLINE")])
rows = []
for chunk in pd.read_csv(ROOT / "data" / "accepted_2007_to_2018Q4.csv", dtype=str, usecols=["id", *engine.raw_fields], chunksize=100_000):
    rows.append(chunk[chunk["id"].str.strip().isin(picks["id"])])
raw = pd.concat(rows).assign(id=lambda d: d["id"].str.strip()).set_index("id").loc[picks["id"]]
real_examples = []
for loan_id, row in raw.iterrows():
    out = engine.score(row[engine.raw_fields].to_dict())
    actual = int(picks.set_index("id").loc[loan_id, "target"])
    real_examples.append({**{k: out[k] for k in ("probability_of_default", "risk_band", "decision", "loan_amount", "term_months", "expected_loss_usd")},
                          "reasons": [r["reason"] for r in out["reasons"]], "strengths": [r["reason"] for r in out["strengths"]],
                          "actual_outcome": "defaulted" if actual else "repaid"})
    print(json.dumps(real_examples[-1], indent=1), "\n")

{
 "probability_of_default": 0.0733,
 "risk_band": "R2",
 "decision": "APPROVE",
 "loan_amount": 14400.0,
 "term_months": 36,
 "expected_loss_usd": 535.65,
 "reasons": [
  "Age of oldest revolving account: 57 months",
  "Debt-to-income ratio: 23.2%",
  "Months since newest revolving account: 23 months",
  "Total balance excluding mortgage: $109,149"
 ],
 "strengths": [
  "Loan term: 36 months",
  "Length of credit history: 4.8 years"
 ],
 "actual_outcome": "repaid"
} 



{
 "probability_of_default": 0.2746,
 "risk_band": "R5",
 "decision": "REVIEW",
 "loan_amount": 24900.0,
 "term_months": 60,
 "expected_loss_usd": 4284.43,
 "reasons": [
  "Loan term: 60 months",
  "Loan amount: $24,900",
  "Accounts opened in the last 24 months: 8",
  "Age of oldest revolving account: 89 months"
 ],
 "strengths": [
  "Annual income: $130,000",
  "Credit score (FICO): 710"
 ],
 "actual_outcome": "repaid"
} 



{
 "probability_of_default": 0.3018,
 "risk_band": "R6",
 "decision": "DECLINE",
 "loan_amount": 20000.0,
 "term_months": 60,
 "expected_loss_usd": 3781.88,
 "reasons": [
  "Loan term: 60 months",
  "Accounts opened in the last 24 months: 10",
  "Loan amount: $20,000",
  "Months since newest bankcard: 2 months"
 ],
 "strengths": [
  "Total credit lines ever: 55",
  "Debt-to-income ratio: 13.4%"
 ],
 "actual_outcome": "repaid"
} 



## 6. Speed
Where the time goes for one application (median of 20 runs).

In [9]:
from src.modeling import reason_codes as rc
from src.modeling import decision as dl
app, missing, _ = engine._prepare(application)
X = engine.transform([app])

def timed(fn, n=20):
    times = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

frame = pd.DataFrame([app]).reindex(columns=engine.raw_fields)
frame = frame.astype(object).where(frame.notna(), np.nan)
steps, x = [], frame
for name, step in engine.preprocessor.steps:
    steps.append({"step": f"preprocessor: {name}", "ms": timed(lambda: step.transform(x))})
    x = step.transform(x)
contrib = engine.booster.predict(X, pred_contrib=True)
steps += [{"step": "LightGBM PD (1,931 trees)", "ms": timed(lambda: engine.booster.predict(X))},
          {"step": "SHAP values (pred_contrib)", "ms": timed(lambda: engine.booster.predict(X, pred_contrib=True))},
          {"step": "calibration + policy + reasons", "ms": timed(lambda: rc.explain(rc.concept_shap(contrib, engine.features).iloc[0], X.iloc[0]))},
          {"step": "TOTAL engine.score()", "ms": timed(lambda: engine.score(application))}]
latency = pd.DataFrame(steps)
te = engine.preprocessor.named_steps["encode"].named_transformers_["te"]
print("Target-encoded categories:", dict(zip(te.feature_names_in_, [len(c) for c in te.named_steps["encode"].categories_])))
latency.round(1)

Target-encoded categories: {'zip_code': 874, 'emp_title': 143539, 'title': 31812}


,step,ms
0,preprocessor: engineer,98.2
1,preprocessor: filter,2.7
2,preprocessor: encode,308.1
3,preprocessor: dedupe,0.5
4,"LightGBM PD (1,931 trees)",1.1
5,SHAP values (pred_contrib),10.8
6,calibration + policy + reasons,1.5
7,TOTAL engine.score(),380.1


Most of the time is the target encoder: on every call scikit-learn rebuilds its lookup table of 143,539 job titles and 31,812 loan titles. About 0.4 s per application is fine for a web demo. If it ever matters, those lookups could be cached as a plain dictionary at start-up, but the engine keeps the exact scikit-learn path because it is verified to reproduce Stage 3 to the last digit.

## 7. Save

In [10]:
example_output = engine.score(application)
(RESULTS / "f4_example_output.json").write_text(json.dumps(example_output, indent=2), encoding="utf-8")
(RESULTS / "f4_real_examples.json").write_text(json.dumps(real_examples, indent=2), encoding="utf-8")
latency.to_csv(RESULTS / "f4_latency.csv", index=False)
contract.to_csv(RESULTS / "f4_input_contract.csv", index=False)
print("Saved:", sorted(p.name for p in RESULTS.iterdir()))

Saved: ['f4_engine_test.json', 'f4_example_output.json', 'f4_input_contract.csv', 'f4_latency.csv', 'f4_real_examples.json']


## 8. Conclusion

In [11]:
print(f"Engine: {manifest['model_name']} v{manifest['version']} | {len(manifest['features'])} features from {len(fields)} raw fields")
print(f"Fresh-session test: {'PASSED' if report['passed'] else 'FAILED'} "
      f"({sum(c['status'] == 'PASS' for c in report['checks'])}/{len(report['checks'])} checks pass)")
print(f"One application: {latency.iloc[-1]['ms']:.0f} ms")
print("Next: Stage 5, open the sealed test set once and score it with this frozen bundle.")

Engine: M3-B-mono-nozip v1.0.0 | 146 features from 65 raw fields
Fresh-session test: PASSED (8/9 checks pass)
One application: 380 ms
Next: Stage 5, open the sealed test set once and score it with this frozen bundle.
